# Lamplighter — training a VAE

A variational autoencoder learns a *smooth latent space*: an *encoder* squeezes
each image into a small distribution (a mean and a variance), and a *decoder*
paints an image back from a point sampled there. Once trained, the latent space
is a map of digit-ness you can wander — decode a random point and get a novel
digit, or walk a straight line between two digits' encodings and watch one
morph into the other. This notebook builds both models on the canvas and trains
them jointly, in this kernel, executing exactly the `train()` the Training tab
shows.

The loop:

1. **Start** a session here → 2. build an **Encoder** (with two *named* outputs:
   `mu` and `logvar`) and a **Decoder** in the **Models** view → 3. register
   MNIST images with `sess.data(X=X)` (no labels) → 4. pick the **VAE** recipe
   in the **Training** tab and press **▶ Run** → 5. pull the trained pair back
   here and explore the latent space.

> This notebook lives in `examples/`; the first cell puts the repo root on
> `sys.path` so `import lamplighter` resolves.

## 1. Start a session

In [ ]:
import sys
from pathlib import Path

# The repo root (this notebook runs with examples/ as its cwd).
sys.path.insert(0, str(Path.cwd().parent.resolve()))

import lamplighter

# A per-notebook autosave path, so this canvas never clobbers
# another example's (they share this working directory).
sess = lamplighter.start(persist=".lamplighter/vae.json")
sess.url

## 2. Build the encoder and decoder

Open the **Models** tab. Use the starting model as the **Encoder** and add a
second for the **Decoder** (the sidebar's **＋**; double-click a name to rename).

**Encoder** (image → a latent *distribution*). The twist: it has **two Output
nodes**, and the recipe finds them **by name** — in each Output's Inspector set
**Name** to `mu` and `logvar` (canvas position doesn't matter, the names do):

> - Input **Shape** `1, 784`
> - → Linear `400` → ReLU — the shared trunk
> - trunk → Linear `16` → Output **Name** `mu`
> - trunk → Linear `16` → Output **Name** `logvar`

**Decoder** (latent point → image). Its Input is the latent size (`16` here),
and it ends in **Sigmoid** so pixels land in `[0, 1]` — which is what the BCE
reconstruction loss expects:

> Input **Shape** `1, 16` → Linear `400` → ReLU → Linear `784` → Sigmoid →
> Output

There's nothing to wire between the models — the recipe composes them
(encode → sample → decode) in the loop; assigning the roles in step 4 is the
composition. The dataset wire into the encoder appears automatically.

## 3. Register MNIST — images only

A VAE reconstructs its input, so it needs no labels. Scale pixels to `[0, 1]`
(BCE's domain — matching the decoder's Sigmoid).

In [ ]:
import torch
from torchvision import datasets

mnist = datasets.MNIST(root="./data", train=True, download=True)
X = (mnist.data.float() / 255.0).view(-1, 784)  # [0, 1] for BCE
torch.manual_seed(0)
X = X[torch.randperm(len(X))[:8000]]  # subsample for a snappy CPU demo

sess.data(X=X)

## 4. Assign the VAE roles and train

Switch to the **Training** tab and choose the **VAE (autoencoder)** recipe.
Assign **Encoder** and **Decoder** to your two models. Unlike a GAN, a VAE
trains *jointly* — one optimizer spans both models, so there's a single
**Learning Rate**. Two knobs are VAE-specific:

- **Beta** — the KL weight. `1.0` is the classic VAE; higher values trade
  reconstruction sharpness for a smoother, more disentangled latent space.
- **Reconstruction Loss** — keep **bce** for `[0, 1]` images.

On the **System** canvas, select the auto-provisioned **Data** node and pick
`X` under **Input(s)** (↻ refresh if it's not listed). Then set **Epochs** —
**~30** (about a minute on CPU for this 8k subset) gives recognizable
reconstructions. Press **▶ Run**: `recon_loss` falls as reconstructions
sharpen, while `kl_loss` typically *rises then settles* — that's the
regularizer pulling the latent space toward a standard normal as the encoder
learns to use it.

In [ ]:
print("epochs trained:", len(sess.history["recon_loss"]))
encoder = sess.models["encoder"].to("cpu").eval()
decoder = sess.models["decoder"].to("cpu").eval()
decoder

## 5. Wander the latent space

The payoff. Two experiments:

**Sample** — decode random points `z ~ N(0, 1)`: novel digits the model has
never seen, painted from pure noise.

**Interpolate** — encode two real digits to their means, walk the straight
line between them, and decode each step. A well-trained VAE morphs one digit
smoothly into the other — the proof the space between digits is meaningful,
which is what separates a VAE from a plain autoencoder.

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    # Random samples from the prior.
    latent = encoder(X[:1]).mu.shape[-1]
    torch.manual_seed(0)
    samples = decoder(torch.randn(8, latent)).reshape(-1, 28, 28)

    # Interpolate between two real digits' encodings (their means).
    a, b = X[0], X[1]
    mu_a, mu_b = encoder(a.unsqueeze(0)).mu, encoder(b.unsqueeze(0)).mu
    steps = torch.linspace(0, 1, 8).unsqueeze(-1)
    walk = decoder(mu_a + steps * (mu_b - mu_a)).reshape(-1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(10, 2.9))
for ax, img in zip(axes[0], samples):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.axis("off")
for ax, img in zip(axes[1], walk):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.axis("off")
axes[0, 0].set_title("samples from z ~ N(0, 1)", loc="left", fontsize=10)
axes[1, 0].set_title("interpolation: digit A → digit B", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Keep the pair

Both models save together — `sess.checkpoint("mnist-vae")` keeps the run in the
app's Checkpoints strip (persisted to `.lamplighter/checkpoints/`, so it
survives a kernel restart), and `sess.save_checkpoint(path)` writes one
self-contained `.pt`. Reload just the decoder by role — that's all you need to
keep generating:

In [ ]:
sess.save_checkpoint("mnist-vae.pt")

decoder2, snapshot = lamplighter.load_checkpoint("mnist-vae.pt", model="decoder")
with torch.no_grad():
    z = torch.randn(1, latent)
    identical = torch.equal(decoder2(z), decoder(z))
print("reloaded decoder matches:", identical)

## 7. Tear down

In [ ]:
lamplighter.stop()